In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
import optuna

# Обучение и сравнение моделей

В этом ноутбуке проводится обучение CatBoost для прогнозирования вероятности дефолта заёмщика.


Основной метрикой качества является ROC-AUC, поскольку она позволяет оценивать способность модели ранжировать клиентов по уровню риска без выбора конкретного порога классификации.

In [2]:
train = pd.read_csv("/kaggle/input/notebooks/alenaglushkova1/02-feature-engeneering/features_train.csv")
test = pd.read_csv("/kaggle/input/notebooks/alenaglushkova1/02-feature-engeneering/features_test.csv")

train_ids = pd.read_csv("/kaggle/input/notebooks/alenaglushkova1/01-data-overview-and-eda/train_ids.csv")["SK_ID_CURR"]
hold_out_ids = pd.read_csv("/kaggle/input/notebooks/alenaglushkova1/01-data-overview-and-eda/test_ids.csv")["SK_ID_CURR"]

application_columns = pd.read_csv("/kaggle/input/notebooks/alenaglushkova1/02-feature-engeneering/application_columns.csv").iloc[:, 0].tolist()

hold = train[train["SK_ID_CURR"].isin(hold_out_ids)].copy()
train_local = train[train["SK_ID_CURR"].isin(train_ids)].copy()

print("Train:", train_local.shape)
print("Holdout:", hold.shape)

Train: (246008, 333)
Holdout: (61503, 333)


# Тестирование модели с разными наборами признаков

Для оценки пользы Feature Engineering сравниваются два набора признаков:

Application Features — исходные и созданные признаки из основной таблицы application_train.

Full Features — признаки основной таблицы и агрегированные характеристики из дополнительных источников: bureau, previous_application, installments_payments, POS_CASH_balance и credit_card_balance.

Для сравнения используется CatBoost с одинаковыми настройками и стратифицированная 5-fold кросс-валидация.

Категориальные признаки обрабатываются средствами CatBoost. Пропущенные значения в них заменяются отдельной категорией Missing.

In [3]:
def run_cross_validation(df, params=None):
    X = df.drop(columns=["SK_ID_CURR", "TARGET"])
    y = df["TARGET"]
    cat_features = X.select_dtypes(
    include=["object", "category"]
    ).columns.tolist()
    
    X = X.copy()
    X[cat_features] = X[cat_features].fillna("Missing").astype(str)
    cat_features = tuple(cat_features)
    base_params = {
        "cat_features": cat_features,
        "eval_metric": "AUC",
        "iterations": 5000,
        "verbose": False,
        "random_seed": 42,
        "task_type": "GPU",
        "devices": "0:1"
    }
    if params is not None:
        base_params.update(params)

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scores = []
    best_iterations = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
    
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]
    
        model = CatBoostClassifier(**base_params)
        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
            eval_set=(X_val, y_val),
            early_stopping_rounds=500,
            verbose=False
        )
        best_iterations.append(model.best_iteration_)

        print(f"Fold {fold}: Best iteration = {model.best_iteration_}")
    
        y_pred = model.predict_proba(X_val)[:, 1]
    
        auc = roc_auc_score(y_val, y_pred)
    
        scores.append(auc)
    
        print(f"Fold {fold}: ROC-AUC = {auc:.6f}")
    print(f"\nСредний ROC-AUC: {np.mean(scores):.6f}")
    print(f"Стандартное отклонение: {np.std(scores):.6f}")
    print(f"Средняя лучшая итерация: {np.mean(best_iterations):.0f}")
    print(f"Стандартное отклонение: {np.std(best_iterations):.0f}")

In [4]:
run_cross_validation(train_local[application_columns])

Default metric period is 5 because AUC is/are not implemented for GPU


Fold 1: Best iteration = 4144
Fold 1: ROC-AUC = 0.765433


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 2: Best iteration = 3643
Fold 2: ROC-AUC = 0.765047


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 3: Best iteration = 4544
Fold 3: ROC-AUC = 0.764829


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 4: Best iteration = 4619
Fold 4: ROC-AUC = 0.767107


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 5: Best iteration = 4218
Fold 5: ROC-AUC = 0.769017

Средний ROC-AUC: 0.766287
Стандартное отклонение: 0.001582
Средняя лучшая итерация: 4234
Стандартное отклонение: 347


In [5]:
run_cross_validation(train_local)

Default metric period is 5 because AUC is/are not implemented for GPU


Fold 1: Best iteration = 4999
Fold 1: ROC-AUC = 0.790030


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 2: Best iteration = 4988
Fold 2: ROC-AUC = 0.790110


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 3: Best iteration = 4976
Fold 3: ROC-AUC = 0.790993


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 4: Best iteration = 4994
Fold 4: ROC-AUC = 0.787987


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 5: Best iteration = 4986
Fold 5: ROC-AUC = 0.790790

Средний ROC-AUC: 0.789982
Стандартное отклонение: 0.001065
Средняя лучшая итерация: 4989
Стандартное отклонение: 8


Добавление агрегированных признаков из дополнительных таблиц увеличило средний ROC-AUC на 0.023695.

Кроме того, стандартное отклонение ROC-AUC между фолдами уменьшилось, что указывает на более стабильные результаты в проведённой кросс-валидации.

Таким образом, информация о предыдущих кредитах, заявках и платежах существенно улучшила способность модели ранжировать заёмщиков по риску дефолта.

Для дальнейших экспериментов выбран полный набор признаков.

# Подбор гиперпараметров

Для настройки CatBoost используется библиотека Optuna с алгоритмом TPE.

In [6]:
X = train_local.drop(columns=["SK_ID_CURR", "TARGET"])
y = train_local["TARGET"]
cat_features = X.select_dtypes(
include=["object", "category"]
).columns.tolist()

X = X.copy()
X[cat_features] = X[cat_features].fillna("Missing").astype(str)
cat_features = tuple(cat_features)

In [7]:
def objective(trial):

    params = {
        "iterations": 5000,
        "depth": trial.suggest_int("depth", 4, 8),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.08, log=True
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1, 10, log=True
        ),
        "random_strength": trial.suggest_float(
            "random_strength", 0.1, 5.0
        ),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": 42,
        "verbose": False,
        "thread_count": -1,
        "task_type": "GPU",
        "devices": "0:1"
    }

    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = []

    for train_idx, val_idx in skf.split(X, y):

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model = CatBoostClassifier(**params)

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
            eval_set=(X_val, y_val),
            early_stopping_rounds=500,
            verbose=False
        )

        predictions = model.predict_proba(X_val)[:, 1]

        auc = roc_auc_score(y_val, predictions)

        scores.append(auc)

    return np.mean(scores)

In [8]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=20, show_progress_bar=True)

[I 2026-11-05 05:09:11,484] A new study created in memory with name: no-name-28317d9b-6ff7-4b2e-9b4d-62f1cb14b20c


  0%|          | 0/20 [00:00<?, ?it/s]

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-11-05 05:12:46,469] Trial 0 finished with value: 0.7876847752824476 and parameters: {'depth': 5, 'learning_rate': 0.07220721098226064, 'l2_leaf_reg': 5.395030966670228, 'random_strength': 3.0334265725654794}. Best is trial 0 with value: 0.7876847752824476.


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-11-05 05:17:44,893] Trial 1 finished with value: 0.7847956413829239 and parameters: {'depth': 4, 'learning_rate': 0.013831748681407519, 'l2_leaf_reg': 1.1430983876313219, 'random_strength': 4.344263114297182}. Best is trial 0 with value: 0.7876847752824476.


... (32 outputs left out)

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-11-05 07:28:31,518] Trial 18 finished with value: 0.7883861400535621 and parameters: {'depth': 8, 'learning_rate': 0.011121635022197275, 'l2_leaf_reg': 1.3194683002638714, 'random_strength': 1.3092550976846224}. Best is trial 3 with value: 0.7888481137081568.


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-11-05 07:40:19,388] Trial 19 finished with value: 0.7889206223021903 and parameters: {'depth': 8, 'learning_rate': 0.014148452306909592, 'l2_leaf_reg': 3.2486891526009836, 'random_strength': 1.670752066149089}. Best is trial 19 with value: 0.7889206223021903.


In [9]:
best_params = study.best_params
print("Лучший ROC-AUC:", study.best_value)
print("Лучшие параметры:")

for key, value in best_params.items():
    print(f"{key}: {value}")

Лучший ROC-AUC: 0.7889206223021903
Лучшие параметры:
depth: 8
learning_rate: 0.014148452306909592
l2_leaf_reg: 3.2486891526009836
random_strength: 1.670752066149089


# Кросс-валидация после подбора гипрепараметров

Для оценки выбранной конфигурации выполняется повторная стратифицированная 5-fold кросс-валидация.

In [10]:
run_cross_validation(train_local, best_params)

Default metric period is 5 because AUC is/are not implemented for GPU


Fold 1: Best iteration = 4977
Fold 1: ROC-AUC = 0.789675


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 2: Best iteration = 4559
Fold 2: ROC-AUC = 0.789731


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 3: Best iteration = 4998
Fold 3: ROC-AUC = 0.790772


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 4: Best iteration = 4855
Fold 4: ROC-AUC = 0.788015


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 5: Best iteration = 4996
Fold 5: ROC-AUC = 0.790603

Средний ROC-AUC: 0.789759
Стандартное отклонение: 0.000979
Средняя лучшая итерация: 4877
Стандартное отклонение: 168


По сравнению с моделью на полном наборе признаков до подбора гиперпараметров ROC-AUC уменьшился с  0.789982 до 0.789759.

Разница небольшая, можно списать на случайность.

# Финальное предсказание

После выбора гиперпараметров модель обучается на всей локальной обучающей выборке, составляющей 80% исходных размеченных данных.

Затем качество оценивается на ранее отложенных 20% клиентов.

In [11]:
def prepare_data(df, feature_cols, cat_features):
    X = df[feature_cols].copy()

    X[cat_features] = (
        X[cat_features]
        .fillna("Missing")
        .astype(str)
    )

    return X

In [12]:
feature_cols = [
    col for col in train_local.columns
    if col not in ["SK_ID_CURR", "TARGET"]
]

cat_features = (
    train_local[feature_cols]
    .select_dtypes(include=["object", "category"])
    .columns.tolist()
)

In [13]:
X_train = prepare_data(train_local, feature_cols, cat_features)
y_train = train_local["TARGET"]

X_hold = prepare_data(hold, feature_cols, cat_features)
y_hold = hold["TARGET"]

In [14]:
model = CatBoostClassifier(
    **best_params,
    iterations=5000,
    cat_features=cat_features,
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    task_type="GPU",
    devices="0:1"
)

model.fit(X_train, y_train)

hold_preds = model.predict_proba(X_hold)[:, 1]

print("Holdout ROC-AUC:", roc_auc_score(y_hold, hold_preds))

Default metric period is 5 because AUC is/are not implemented for GPU


Holdout ROC-AUC: 0.7932466278417065


In [15]:
model.save_model("catboost_holdout.cbm")

После локальной оценки создаётся отдельная финальная модель, которая обучается на всех 307 511 размеченных наблюдениях application_train и тестируется на выборке Kaggle.

In [16]:
full_train = pd.read_csv("/kaggle/input/notebooks/alenaglushkova1/02-feature-engeneering/features_train.csv")

X_full = prepare_data(full_train, feature_cols, cat_features)
y_full = full_train["TARGET"]

final_model = CatBoostClassifier(
    **best_params,
    cat_features=cat_features,
    iterations=5000,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    thread_count=-1,
    task_type="GPU",
    devices="0:1"
)

final_model.fit(X_full, y_full)

Default metric period is 5 because AUC is/are not implemented for GPU


CatBoostClassifier(cat_features=['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE'], depth=8, devices='0:1', eval_metric='AUC', iterations=5000, l2_leaf_reg=3.2486891526009836, learning_rate=0.014148452306909592, loss_function='Logloss', random_seed=42, random_strength=1.670752066149089, task_type='GPU', verbose=False)

In [17]:
X_test = prepare_data(test, feature_cols, cat_features)

test_preds = final_model.predict_proba(X_test)[:, 1]

In [18]:
submission = pd.DataFrame({
    "SK_ID_CURR": test["SK_ID_CURR"],
    "TARGET": test_preds
})

submission.to_csv("submission.csv", index=False)

submission.head()

,SK_ID_CURR,TARGET
0,100001,0.045519
1,100005,0.179167
2,100013,0.032070
3,100028,0.032417
4,100038,0.127920


# Выводы

В ходе моделирования были проведены эксперименты с различными наборами признаков, выполнен подбор гиперпараметров CatBoost и получены предсказания для тестовой выборки Kaggle.

### 1. Влияние Feature Engineering

Использование дополнительных данных о кредитной истории и предыдущих платежах позволило увеличить средний ROC-AUC с **0.766287 до 0.789982**.

Прирост составил **0.023695**, что подтверждает полезность агрегированных признаков из дополнительных источников.

Именно расширение набора признаков обеспечило основное улучшение качества модели.

### 2. Результаты настройки гиперпараметров

Подбор гиперпараметров с помощью Optuna позволил уменьшить средний ROC-AUC до **0.789759**. Что можно считать случайным незначительным изменением.


Таким образом, в проведённых экспериментах настройка гиперпараметров не оказала  влияния на качество, в отличие от Feature Engineering.

### 3. Стабильность результатов

Результаты 5-fold кросс-валидации после настройки гиперпараметров:

- Средний ROC-AUC: **0.789759**.
- Стандартное отклонение: **0.000979**.

Небольшое стандартное отклонение свидетельствует об относительно стабильном качестве модели на использованных разбиениях обучающих данных.

При этом кросс-валидация проводилась со случайным стратифицированным разделением, поэтому не позволяет оценить устойчивость модели к изменениям данных во времени.

### 4. Качество на отложенной выборке

На локальной отложенной выборке получено значение **ROC-AUC = 0.793246**.

Результат близок к среднему значению кросс-валидации, что подтверждает согласованность полученных оценок в рамках проведённых экспериментов.

### 5. Итоговая модель

Для финального решения выбран CatBoost с расширенным набором признаков и гиперпараметрами, найденными с помощью Optuna.

После локальной оценки модель обучается на всех доступных размеченных данных и используется для прогнозирования вероятностей дефолта на тестовой выборке Kaggle.

### Заключение

Проведённые эксперименты показали, что **создание информативных признаков на основе дополнительных источников данных оказало наиболее заметное влияние на качество кредитного скоринга**.

Подбор гиперпараметров позволил получить небольшое дополнительное улучшение, а результаты кросс-валидации и локальной оценки оказались близкими.

В следующем ноутбуке проводится подробная оценка качества вероятностных прогнозов, анализ порогов классификации и интерпретация модели с помощью SHAP.